In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
# Pipeline Configuration

PIPELINE_NAME = "Gold FUNNEL ANALYSIS"

SOURCE_TABLE = GOLD_FACT_SALES
TARGET_TABLE = GOLD_FUNNEL_ANALYSIS

RUN_ID = generate_run_id()
START_TIME = datetime.now()

In [0]:
print("GOLD SALES FUNNEL ANALYSIS PIPELINE")

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID : {RUN_ID}")
print(f"Target : {TARGET_TABLE}")

GOLD SALES FUNNEL ANALYSIS PIPELINE
Pipeline : Gold FUNNEL ANALYSIS
Run ID : 3a2d4cc4-39d9-4767-9f34-c1257e2df678
Target : retailmart.gold.funnel_analysis


In [0]:
fact_sales_df = spark.table(GOLD_FACT_SALES)
display(fact_sales_df.limit(10))
fact_sales_df.printSchema()

order_id,order_item_id,order_status,order_purchase_timestamp,order_year,order_month,revenue_month,order_delivered_customer_date,delivery_duration_days,customer_id,customer_city,customer_state,product_id,product_category_name,price,freight_value,total_item_value,payment_type,payment_installments,total_payment_value
ORD_0000001,1,delivered,2023-06-15T14:30:00.000Z,2023,6,2023-06,2023-06-19T14:30:00.000Z,4,CUST_006571,Belo Horizonte,MG,PROD_001950,books,1754.7,20.17,1774.87,multiple,12,2807.85
ORD_0000002,1,cancelled,2021-06-12T11:02:00.000Z,2021,6,2021-06,null,null,CUST_006956,Aracaju,SE,PROD_000989,food,2105.2,45.48,2150.68,voucher,1,805.73
ORD_0000003,1,delivered,2022-03-31T19:38:00.000Z,2022,3,2022-03,2022-04-14T19:38:00.000Z,14,CUST_008373,Joao Pessoa,PB,PROD_000254,furniture,1627.94,78.29,1706.23,credit_card,6,508.88
ORD_0000004,1,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001327,music,65.22,31.08,96.3,boleto,3,1624.71
ORD_0000004,2,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_002714,computers,946.39,44.43,990.82,boleto,3,1624.71
ORD_0000004,3,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001507,home_appliances,157.96,55.93,213.89,boleto,3,1624.71
ORD_0000005,1,shipped,2022-08-27T07:46:00.000Z,2022,8,2022-08,null,null,CUST_002810,Manaus,AM,PROD_001872,fashion,1681.83,14.0,1695.83,credit_card,3,1592.91
ORD_0000006,1,invoiced,2023-05-26T16:27:00.000Z,2023,5,2023-05,null,null,CUST_007867,Recife,PE,PROD_001518,garden,856.74,64.83,921.57,credit_card,1,495.97
ORD_0000007,1,processing,2021-09-21T15:50:00.000Z,2021,9,2021-09,null,null,CUST_004827,Campo Grande,MS,PROD_001504,music,373.8,66.88,440.68,credit_card,2,1738.92
ORD_0000008,1,delivered,2021-01-19T14:24:00.000Z,2021,1,2021-01,2021-01-28T14:24:00.000Z,9,CUST_005137,Porto Velho,RO,PROD_001133,health,899.98,23.95,923.93,voucher,1,1107.69


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_month: string (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- total_payment_value: double (nullable = true)



In [0]:
print(f"Total Records : {fact_sales_df.count()}")

Total Records : 86328


In [0]:
#Explore order status
spark.sql(f"""
SELECT
    order_status,
    COUNT(DISTINCT order_id) AS total_orders
FROM {GOLD_FACT_SALES}
GROUP BY order_status
ORDER BY total_orders DESC
""").show(truncate=False)

+------------+------------+
|order_status|total_orders|
+------------+------------+
|delivered   |24966       |
|invoiced    |6421        |
|shipped     |6288        |
|cancelled   |6193        |
|processing  |6132        |
+------------+------------+



In [0]:
# Revenue by order status
spark.sql(f"""
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(SUM(total_item_value),2) AS total_revenue,
    ROUND(AVG(total_item_value),2) AS avg_order_value
FROM {GOLD_FACT_SALES}
GROUP BY order_status
ORDER BY total_revenue DESC
""").show(truncate=False)

+------------+------------+-------------+---------------+
|order_status|total_orders|total_revenue|avg_order_value|
+------------+------------+-------------+---------------+
|delivered   |43307       |5.639313836E7|1302.17        |
|invoiced    |11092       |1.434830374E7|1293.57        |
|shipped     |10777       |1.406631611E7|1305.22        |
|cancelled   |10602       |1.375534084E7|1297.43        |
|processing  |10550       |1.372812249E7|1301.24        |
+------------+------------+-------------+---------------+



In [0]:
# Checking timestamps
fact_sales_df.columns

['order_id',
 'order_item_id',
 'order_status',
 'order_purchase_timestamp',
 'order_year',
 'order_month',
 'revenue_month',
 'order_delivered_customer_date',
 'delivery_duration_days',
 'customer_id',
 'customer_city',
 'customer_state',
 'product_id',
 'product_category_name',
 'price',
 'freight_value',
 'total_item_value',
 'payment_type',
 'payment_installments',
 'total_payment_value']

In [0]:
spark.sql(f"""

CREATE OR REPLACE TABLE {TARGET_TABLE} AS

WITH status_summary AS
(
    SELECT
        order_status,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(total_item_value),2) AS total_revenue,
        ROUND(AVG(total_item_value),2) AS average_order_value,
        ROUND(AVG(delivery_duration_days),2) AS average_delivery_days

    FROM {SOURCE_TABLE}
    GROUP BY order_status
),
overall_summary AS
(
    SELECT
        SUM(total_orders) AS overall_orders,
        SUM(total_revenue) AS overall_revenue
    FROM status_summary
),

sales_funnel AS
(
    SELECT

        CASE
            WHEN ss.order_status = 'processing' THEN 1
            WHEN ss.order_status = 'invoiced' THEN 2
            WHEN ss.order_status = 'shipped' THEN 3
            WHEN ss.order_status = 'delivered' THEN 4
            WHEN ss.order_status = 'cancelled' THEN 5
            ELSE 6
        END AS stage_rank,
        ss.order_status AS funnel_stage,
        ss.total_orders,
        ss.total_revenue,
        ss.average_order_value,
        ss.average_delivery_days,

        ROUND(
            (ss.total_orders * 100.0) / os.overall_orders,
            2
        ) AS order_percentage,

        ROUND(
            (ss.total_revenue * 100.0) / os.overall_revenue,
            2
        ) AS revenue_percentage
    FROM status_summary ss
    CROSS JOIN overall_summary os
)

SELECT
    stage_rank,
    funnel_stage,
    total_orders,
    total_revenue,
    average_order_value,
    average_delivery_days,
    order_percentage,
    revenue_percentage
FROM sales_funnel
ORDER BY stage_rank

""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
sales_funnel_df = spark.table(TARGET_TABLE)
display(sales_funnel_df)

stage_rank,funnel_stage,total_orders,total_revenue,average_order_value,average_delivery_days,order_percentage,revenue_percentage
1,processing,6132,1.372812249E7,1301.24,null,12.26,12.23
2,invoiced,6421,1.434830374E7,1293.57,null,12.84,12.78
3,shipped,6288,1.406631611E7,1305.22,null,12.58,12.53
4,delivered,24966,5.639313836E7,1302.17,10.08,49.93,50.22
5,cancelled,6193,1.375534084E7,1297.43,null,12.39,12.25


In [0]:
# Validation1
rows_written = sales_funnel_df.count()
print(f"Rows Written : {rows_written}")

Rows Written : 5


In [0]:
# Validation2
sales_funnel_df.printSchema()

root
 |-- stage_rank: integer (nullable = true)
 |-- funnel_stage: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- average_delivery_days: double (nullable = true)
 |-- order_percentage: decimal(27,2) (nullable = true)
 |-- revenue_percentage: double (nullable = true)



In [0]:
# Validation3
sales_funnel_df.describe().show()

+-------+------------------+------------+-----------------+--------------------+-------------------+---------------------+------------------+------------------+
|summary|        stage_rank|funnel_stage|     total_orders|       total_revenue|average_order_value|average_delivery_days|  order_percentage|revenue_percentage|
+-------+------------------+------------+-----------------+--------------------+-------------------+---------------------+------------------+------------------+
|  count|                 5|           5|                5|                   5|                  5|                    1|                 5|                 5|
|   mean|               3.0|        NULL|          10000.0|2.2458244308000002E7|           1299.926|                10.08|         20.000000|            20.002|
| stddev|1.5811388300841898|        NULL|8366.959035396312|1.8971875137606923E7|  4.511577329493564|                 NULL|16.732801618378197|16.893882620641115|
|    min|                 1|   can

In [0]:
# Validation4
assert rows_written > 0

In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_TABLE,
    target=TARGET_TABLE,
    rows_read=fact_sales_df.count(),
    rows_written=rows_written,
    duplicate_count=0,
    start_time=START_TIME,
    status="SUCCESS"
)

LOAD REPORT
Pipeline        : Gold FUNNEL ANALYSIS
Run ID          : 3a2d4cc4-39d9-4767-9f34-c1257e2df678
Source          : retailmart.gold.fact_sales
Target          : retailmart.gold.funnel_analysis
Rows Read       : 86328
Rows Written    : 5
Duplicate Rows  : 0
Start Time      : 2026-07-19 10:22:26.504595
End Time        : 2026-07-19 10:22:48.813041
Duration (sec)  : 22.31
Status          : SUCCESS


# Engineering Observations
• Implemented Sales Funnel Analysis using multiple SQL CTEs.
• Calculated order-level KPIs for each funnel stage.
• Computed order and revenue contribution percentages.
• Introduced stage_rank to preserve business-defined funnel sequence.
• Produced a Gold-layer analytics table suitable for dashboards and reporting.